# 13 · From pixels to data – invoices, receipts and Content Understanding

OCR gives text. Business needs **fields**: vendor, invoice number, total, line items – with a **confidence** per value so you know what to double-check.

| 🧰 | 🎯 Use | 🧠 Engine |
|---|---|---|
| **Document Intelligence prebuilt models** (`prebuilt-invoice`, `prebuilt-receipt`, ...) | common, structured document types; fast, deterministic, proven | specialised vision models |
| **Content Understanding** | your own schema, varied or complex documents, inferred fields, summaries, classification | LLM + OCR + grounding, needs model deployments |

Then we make it an **agent tool**: the agent decides *when* to read a document, the service does the reading. Deep dive: [docs/foundry-tools.md](../docs/foundry-tools.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *

import pandas as pd

## 1 · `prebuilt-invoice`

In [ ]:
inv = extract_invoice("data/docs/invoice-pym-4471.pdf")
pd.DataFrame({"value": inv["fields"], "confidence": pd.Series(inv["confidence"])})

In [ ]:
pd.DataFrame(inv["items"])

### Trust, but verify
Confidence below a threshold goes to a human; and arithmetic is checked **in code**, not by the model.

In [ ]:
THRESHOLD = 0.8
low = {k: c for k, c in inv["confidence"].items() if c < THRESHOLD}
print("needs human review:", low or "nothing")

items_total = sum(float(i["amount"] or 0) for i in inv["items"])
sub = float(inv["fields"]["SubTotal"] or 0)
print(f"line items {items_total:,.2f} vs subtotal {sub:,.2f} ->", "OK" if abs(items_total - sub) < 0.01 else "MISMATCH")

## 2 · `prebuilt-receipt`
Same call, another model id.

In [ ]:
rec = di_analyze("prebuilt-receipt", "data/docs/receipt-dummy-cafe.png")
doc = rec.documents[0]
for k in ("MerchantName", "TransactionDate", "Subtotal", "TotalTax", "Total"):
    f = doc.fields.get(k)
    print(f"{k:16} {field_value(f)}   (conf {f.get('confidence') if f else '-'})")

## 3 · Content Understanding
An **analyzer** defines what to extract. Prebuilt analyzers exist (`prebuilt-invoice`, `prebuilt-documentSearch` for Markdown + figures). CU uses **LLM deployments in your Foundry resource**, so they must exist and be registered as *defaults*:

```bash
azd env set STARK_ENABLE_CU true && azd provision      # deploys gpt-4.1, gpt-4.1-mini, text-embedding-3-large
```

In [ ]:
CU = cfg("STARK_ENABLE_CU", "false").lower() == "true"
print("Content Understanding models deployed:", CU)
if CU:
    # one-time: tell CU which of your deployments to use for each model family
    cu_client().update_defaults(model_deployments={
        "gpt-4.1": "gpt-4-1", "gpt-4.1-mini": "gpt-4-1-mini", "text-embedding-3-large": "text-embedding-3-large"})

In [ ]:
if CU:
    r = cu_analyze("prebuilt-invoice", "data/docs/invoice-oscorp-9930.pdf")
    display(pd.DataFrame(cu_fields(r)))

### A custom analyzer with your own schema
Three field methods: **extract** (literal text), **generate** (the model writes it), **classify** (pick from a list). `estimate_source_and_confidence` adds grounding.

In [ ]:
import time
if CU:
    from azure.ai.contentunderstanding.models import (ContentAnalyzer, ContentAnalyzerConfig, ContentFieldSchema,
                                                      ContentFieldDefinition, ContentFieldType, GenerationMethod)
    analyzer_id = f"stark_invoice_{int(time.time())}"
    schema = ContentFieldSchema(name="stark_invoice", description="Fields the workshop cares about", fields={
        "vendor": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.EXTRACT,
                                         description="Name of the selling company", estimate_source_and_confidence=True),
        "total": ContentFieldDefinition(type=ContentFieldType.NUMBER, method=GenerationMethod.EXTRACT,
                                        description="Total amount due in USD", estimate_source_and_confidence=True),
        "risk_note": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.GENERATE,
                                            description="One sentence: anything unusual a finance reviewer should check"),
        "category": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.CLASSIFY,
                                           description="Spend category", enum=["materials", "freight", "services", "other"]),
    })
    cu_client().begin_create_analyzer(analyzer_id=analyzer_id, resource=ContentAnalyzer(
        base_analyzer_id="prebuilt-document", description="Stark invoice triage",
        config=ContentAnalyzerConfig(enable_layout=True, enable_ocr=True, return_details=True),
        field_schema=schema,
        models={"completion": "gpt-4.1", "embedding": "text-embedding-3-large"},
    )).result()
    r = cu_analyze(analyzer_id, "data/docs/invoice-oscorp-9930.pdf")
    display(pd.DataFrame(cu_fields(r)))
    cu_client().delete_analyzer(analyzer_id=analyzer_id)
    print("analyzer deleted")

## 4 · Make it an agent tool
The agent should not read PDFs in its head. Give it **two functions**: `extract_invoice` (Document Intelligence) and `check_approval_limit` (business rule). It decides when to call them.

In [ ]:
import json
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool

LIMITS = {"default": 5000, "pepper potts": 250000, "tony stark": 1000000}
def check_approval_limit(amount: float, approver: str) -> dict:
    limit = LIMITS.get(approver.strip().lower(), LIMITS["default"])
    return {"approver": approver, "limit_usd": limit, "within_limit": amount <= limit}

IMPL = {"extract_invoice": lambda path: extract_invoice(path), "check_approval_limit": check_approval_limit}
tools = [
    FunctionTool(name="extract_invoice", description="Read an invoice file (path relative to the repo, e.g. data/docs/invoice-pym-4471.pdf) and return its fields, line items and confidences.",
                 parameters={"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"], "additionalProperties": False}, strict=True),
    FunctionTool(name="check_approval_limit", description="Check whether an amount in USD is within an approver's limit.",
                 parameters={"type": "object", "properties": {"amount": {"type": "number"}, "approver": {"type": "string"}},
                             "required": ["amount", "approver"], "additionalProperties": False}, strict=True),
]
oai = openai_client()
agent = project().agents.create_version(agent_name="friday-docs-nb13", definition=PromptAgentDefinition(
    model=model("chat"), instructions="You are F.R.I.D.A.Y., the finance assistant. Use the tools for every fact. Answer in at most four sentences.", tools=tools))

conv = oai.conversations.create()
resp = oai.responses.create(conversation=conv.id, input="Read data/docs/invoice-oscorp-9930.pdf. Can Pepper Potts approve it?", extra_body=agent_ref(agent))
for _ in range(6):
    calls = [i for i in resp.output if i.type == "function_call"]
    if not calls:
        break
    outs = []
    for c in calls:
        result = IMPL[c.name](**json.loads(c.arguments))
        print("  tool:", c.name, "->", str(result)[:100])
        outs.append({"type": "function_call_output", "call_id": c.call_id, "output": json.dumps(result, default=str)})
    resp = oai.responses.create(conversation=conv.id, input=outs, extra_body=agent_ref(agent))
print(resp.output_text)

## Clean up

In [ ]:
oai.conversations.delete(conv.id)
delete_agent("friday-docs-nb13")

## Recap
| Need | Use |
|---|---|
| Standard invoice/receipt/ID, speed, determinism | Document Intelligence prebuilt model |
| Your own fields, varied layouts, summaries, classification | Content Understanding custom analyzer |
| LLM answers over documents | agent + tool that calls the service |
| Always | confidence thresholds + arithmetic checks in code |

> 💡 Next: Act V – the warning. [14 · Ultron](14_ultron_guardrails_redteam.ipynb).